<a href="https://colab.research.google.com/github/S3oudd/eplTopScorers/blob/main/Machine_Learning_EPL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#PD Stands for player Data
import pandas
import requests
import numpy


pd = pandas.read_csv("Scorers.csv")
pd = pd.sort_values("Gls")
pd = pd.reset_index(drop=True)

pd

In [ ]:

def target(player):
  player["target"] = player["Gls"].shift(-1)
  return player

pd = pd.groupby("Player", group_keys=False).apply(target)



In [ ]:
#pd[pd["Player"] == "Ollie Watkins"]


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import RidgeClassifier

rr = RidgeClassifier(alpha = 1)
split = TimeSeriesSplit(n_splits = 3)

sfs = SequentialFeatureSelector(rr, n_features_to_select=15, direction = "forward", cv=split)


In [ ]:
removed_columns = ["Player", "Gls", "target", "Pos", "Age", 'Starts',"Team", "Nation"]
selected_columns = pd.columns[~pd.columns.isin(removed_columns)]

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
pd[selected_columns] = scaler.fit_transform(pd[selected_columns])

In [ ]:
pd

In [ ]:
sfs.fit(pd[selected_columns], pd["target"])

In [ ]:
predictors = list(selected_columns[sfs.get_support()])
predictors

In [ ]:
def test(data, model, predictors, start=2, step=1):
  totalPredictions = []

  Gls_values = sorted(data["Gls"].unique())
  xG = sorted(data["xG_90"].unique())

  for i in range(start, len(Gls_values), step):
    Gls = Gls_values[i]

    data_test = data[data["Gls"] == Gls]
    data_train = data[data["Gls"] != Gls]

    model.fit(data_train[predictors], data_train["target"])

    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index)
    preds.name = "predictions"

    combined = pd.concat([test["target"], preds], axis=1)
    combined.columns = ["target", "predictions"]

    totalPredictions.append(combined)
    return combined

In [ ]:
predictions = test(pd, rr, predictors)

In [ ]:
predictions

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns


data = pd.read_csv("Scorers.csv")

print(data.head())

correlation_matrix = data[['Age', 'Gls']].corr()
print("Correlation Matrix:")
print(correlation_matrix)


predictors = ['Age']
target = 'Gls'

X = data[predictors]
y = data[target]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print("\nLinear Regression Summary:")
print(model.summary())


plt.figure(figsize=(10, 6))
sns.regplot(x='Age', y='Gls', data=data, scatter_kws={'s':50}, line_kws={'color':'red'})
plt.title('Scatter Plot of Age vs. Goals Scored')
plt.xlabel('Age')
plt.ylabel('Goals Scored')
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

data = pd.read_csv('Scorers.csv')

print(data.head())

predictors = ['Age', 'xG']
target = 'Gls'

X = data[predictors]
y = data[target]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()


data['Predicted_Gls'] = model.predict(X)

top_players = data[['Player', 'Age', 'xG', 'Gls', 'Predicted_Gls']].sort_values(by='Predicted_Gls', ascending=False)

print("\nPlayers with Highest Predicted Goals:")
print(top_players.head(10))

plt.figure(figsize=(12, 8))
sns.scatterplot(x='Predicted_Gls', y='Gls', data=data, hue='Pos', palette='viridis', s=100)
plt.title('Predicted vs Actual Goals Scored')
plt.xlabel('Predicted Goals')
plt.ylabel('Actual Goals')
plt.legend(title='Position')
plt.show()
